# 04 — LegalBert-pt + LoRA Ponderado (opcional, requer GPU)

Fine-tuning do `dominguesm/legal-bert-base-cased-ptbr` com **LoRA** sobre
**`VOTO_LIMPO`**, com pesos de classe (cross-entropy ponderada ou Focal Loss).
Mesmos guardrails dos notebooks anteriores: nenhuma feature vem do SUMARIO,
e o corpus já passou pelo gate `auditar_vazamento` no notebook 01.

**Requer GPU** (Colab: Runtime → Alterar tipo de ambiente → T4). Sem GPU, a
célula de configuração reduz drasticamente o escopo (menos folds, menos
épocas, sequência mais curta) só para **validar o fluxo** — os números dessa
execução reduzida não são resultado final.

**Custo esperado (com T4):** ~15–20 min por fold × 5 folds ≈ 1,5–2h para
`n_splits=5`. Ajuste `N_SPLITS` na célula de configuração se quiser algo mais rápido.

**Saída:** `resultados/metricas_legalbert.json` + tabela comparativa com
baseline (02) e TextCNN (03), se os JSONs desses notebooks já existirem.


## 1. Setup


In [2]:
import os, sys, subprocess
REPO_DIR = os.environ.get('REPO_DIR', '/content/deep-acordao-tcu2')
REPO_URL = 'https://github.com/bsousa7/deep-acordao-tcu2.git'
BRANCH = os.environ.get('BRANCH', 'claude/deep-acordao-tcu-refactor-yyjfr3')
if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
    subprocess.run(['git', 'clone', REPO_URL, '--branch', BRANCH, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('repo em', REPO_DIR)


repo em /content/deep-acordao-tcu2


In [3]:
%pip -q install "transformers>=4.44" "peft>=0.13" "datasets>=2.20" accelerate scikit-learn pyarrow scipy nltk
import torch
print('deps ok | torch', torch.__version__)


deps ok | torch 2.11.0+cpu


## 2. Checagem de GPU


In [4]:
import torch
USAR_GPU = torch.cuda.is_available()
if USAR_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('AVISO: GPU não detectada.')
    print('Ative em Runtime > Alterar tipo de ambiente de execução > T4.')
    print('Sem GPU, a célula de configuração a seguir reduz o escopo para apenas'
          ' validar o fluxo (números NÃO são finais).')


AVISO: GPU não detectada.
Ative em Runtime > Alterar tipo de ambiente de execução > T4.
Sem GPU, a célula de configuração a seguir reduz o escopo para apenas validar o fluxo (números NÃO são finais).


## 3. Configuração

Com GPU: escopo completo (`n_splits=5`, `max_length=512`, `epochs=5`, `batch_size=16`),
espelhando os parâmetros testados em `CLAUDE.md`.

Sem GPU: reduzido automaticamente (`n_splits=3`, `max_length=128`, `epochs=1`,
`batch_size=8`) — só para checar que o código roda ponta a ponta.


In [5]:
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

RANDOM_STATE = 42
LOSS = 'weighted_ce'    # 'weighted_ce' (padrão) ou 'focal' (desbalanceamento severo)
FOCAL_GAMMA = 2.0
USAR_LORA = True

if USAR_GPU:
    N_SPLITS, MAX_LENGTH, EPOCHS, BATCH_SIZE = 5, 512, 5, 16
    print('GPU detectada -> config completa (n_splits=5, max_length=512, epochs=5)')
else:
    N_SPLITS, MAX_LENGTH, EPOCHS, BATCH_SIZE = 3, 128, 1, 8
    print('SEM GPU -> config reduzida apenas para validar o fluxo'
          ' (n_splits=3, max_length=128, epochs=1).')

BASE = Path(REPO_DIR)

# Mesma lógica de persistência dos notebooks 01/02/03: prefere o Google Drive
# se já houver dados lá (gerados pelo 01), com fallback para o clone local.
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive') / 'deep-acordao-tcu2'
except Exception as _e:
    DRIVE_ROOT = None
    print(f'Google Drive indisponível (fora do Colab?): {_e}')

_drive_interim = (DRIVE_ROOT / 'data' / 'interim') if DRIVE_ROOT else None
PERSIST_BASE = DRIVE_ROOT if (_drive_interim and _drive_interim.exists()) else BASE

DATA_INTERIM = PERSIST_BASE / 'data' / 'interim'
RESULTADOS = PERSIST_BASE / 'resultados'
FIGURAS = RESULTADOS / 'figuras'; FIGURAS.mkdir(parents=True, exist_ok=True)

print(f'Lendo/gravando dados em: {PERSIST_BASE}')


SEM GPU -> config reduzida apenas para validar o fluxo (n_splits=3, max_length=128, epochs=1).
Mounted at /content/drive
Lendo/gravando dados em: /content/drive/MyDrive/deep-acordao-tcu2


## 4. Carrega o corpus (gerado pelo notebook 01)


In [6]:
import pandas as pd

df = pd.read_parquet(DATA_INTERIM / 'acordaos_rotulados.parquet')
print('corpus n =', len(df))
print(df['LABEL'].value_counts().to_string())

contagens = df['LABEL'].value_counts()
classe_minima = contagens.min()
if classe_minima < N_SPLITS:
    print(f"AVISO: classe mais rara tem {classe_minima} amostras — reduzindo"
          f" N_SPLITS de {N_SPLITS} para {classe_minima}.")
    N_SPLITS = int(classe_minima)


corpus n = 3644
LABEL
Irregular               3310
Regular com Ressalva     241
Regular                   93


## 5. Fine-tuning LoRA K-Fold — VOTO_LIMPO ponderado

`WeightedTrainer` (cross-entropy ponderada) por padrão. Use `LOSS='focal'` na
célula de configuração para a variante Focal Loss (Lin et al., 2017).


In [7]:
import subprocess
subprocess.run(["pip", "uninstall", "-y", "torchao"], check=True)

# limpa qualquer import parcial/quebrado de peft nesta sessão
import sys
for mod in list(sys.modules):
    if mod.startswith('peft') or mod.startswith('torchao'):
        del sys.modules[mod]

print('torchao removido e cache de imports limpo — tente o K-Fold de novo.')

torchao removido e cache de imports limpo — tente o K-Fold de novo.


In [8]:
from src.modelos.transformer import kfold

res_legalbert = kfold(
    df, campo='VOTO_LIMPO', n_splits=N_SPLITS,
    loss=LOSS, focal_gamma=FOCAL_GAMMA, usar_lora=USAR_LORA,
    epochs=EPOCHS, max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
    output_base=str(RESULTADOS / 'modelos_legalbert'),
)
print(f"F1-macro = {res_legalbert['mean_f1']:.4f}  IC95={res_legalbert['ci_95']}")
print(f"Acurácia média = {res_legalbert['mean_acc']:.4f}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dominguesm/legal-bert-base-cased-ptbr
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,811 || all params: 126,276,870 || trainable%: 0.7031


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,1.002531,1.055905,0.317278,0.908093
2,0.987429,1.089545,0.353260,0.893004
3,1.071087,1.070266,0.273009,0.506173
4,1.039783,1.076139,0.352835,0.866941


...

trainable params: 887,811 || all params: 126,276,870 || trainable%: 0.7031


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,1.040490,1.057767,0.158689,0.245879
2,1.083948,1.037839,0.317255,0.907967
3,0.980010,1.028132,0.317003,0.906593
4,1.036756,1.023969,0.326544,0.896978
5,1.047119,1.026101,0.320514,0.814560


Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
1.047119,1.026101,5,0.320514,0.814560


F1-macro = 0.3294  IC95=(0.31247966550674167, 0.3463498240386379)
Acurácia média = 0.8570


## 6. (Opcional) Focal Loss para contraste

Só execute esta célula se quiser comparar `weighted_ce` vs `focal` — **dobra o
tempo de GPU** (mais um K-Fold completo). Deixe comentado para pular.


In [9]:
RODAR_FOCAL = False  # mude para True para rodar o contraste

res_legalbert_focal = None
if RODAR_FOCAL:
    res_legalbert_focal = kfold(
        df, campo='VOTO_LIMPO', n_splits=N_SPLITS,
        loss='focal', focal_gamma=FOCAL_GAMMA, usar_lora=USAR_LORA,
        epochs=EPOCHS, max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
        output_base=str(RESULTADOS / 'modelos_legalbert_focal'),
    )
    print(f"F1-macro (focal) = {res_legalbert_focal['mean_f1']:.4f}"
          f"  IC95={res_legalbert_focal['ci_95']}")
else:
    print('RODAR_FOCAL=False — pulando contraste com Focal Loss.')


RODAR_FOCAL=False — pulando contraste com Focal Loss.


## 7. Comparação com baseline (02) e TextCNN (03)

Carrega os JSONs desses notebooks (se existirem no mesmo `PERSIST_BASE`) e monta
uma tabela única de F1-macro.


In [15]:
import json
from src.avaliacao.metricas import comparar_modelos

todos_resultados = {'LegalBert-pt + LoRA (' + LOSS + ')': res_legalbert}
if res_legalbert_focal is not None:
    todos_resultados['LegalBert-pt + LoRA (focal)'] = res_legalbert_focal

caminho_baseline = RESULTADOS / 'metricas_baseline.json'
if caminho_baseline.exists():
    m = json.loads(caminho_baseline.read_text())
    todos_resultados['TF-IDF + LogReg (balanced)'] = m['kfold_5x_balanced']
else:
    print(f'Aviso: {caminho_baseline} não encontrado — rode o notebook 02 antes.')

caminho_textcnn = RESULTADOS / 'metricas_textcnn.json'
if caminho_textcnn.exists():
    m = json.loads(caminho_textcnn.read_text())
    todos_resultados['TextCNN'] = m['kfold_5x']
else:
    print(f'Aviso: {caminho_textcnn} não encontrado — rode o notebook 03 antes.')

tabela = comparar_modelos(todos_resultados)
print(tabela.to_string(index=False))
tabela.to_csv(RESULTADOS / 'comparacao_modelos.csv', index=False)


                     Modelo/Campo  F1-macro  ± std             IC95  Acurácia
       TF-IDF + LogReg (balanced)    0.4910 0.0705 [0.4035, 0.5785]    0.9125
                          TextCNN    0.3674 0.0000                —    0.8850
LegalBert-pt + LoRA (weighted_ce)    0.3294 0.0136 [0.3125, 0.3463]    0.8570


## 8. Persistência


In [16]:
from src.avaliacao.metricas import salvar_json

saida = {
    'modelo': 'LegalBert-pt + LoRA',
    'feature': 'VOTO_LIMPO',
    'loss': LOSS,
    'usar_lora': USAR_LORA,
    'usar_gpu': USAR_GPU,
    'hiperparametros': {
        'n_splits': N_SPLITS, 'max_length': MAX_LENGTH,
        'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
    },
    'kfold': {
        'mean_f1': res_legalbert['mean_f1'],
        'std_f1': res_legalbert['std_f1'],
        'ci_95': list(res_legalbert['ci_95']),
        'mean_acc': res_legalbert['mean_acc'],
        'fold_scores': res_legalbert['fold_scores'],
    },
}
if res_legalbert_focal is not None:
    saida['kfold_focal'] = {
        'mean_f1': res_legalbert_focal['mean_f1'],
        'ci_95': list(res_legalbert_focal['ci_95']),
        'mean_acc': res_legalbert_focal['mean_acc'],
    }

salvar_json(saida, RESULTADOS / 'metricas_legalbert.json')
print(f'OK — persistido em: {PERSIST_BASE}')
if not USAR_GPU:
    print('LEMBRETE: esta execução rodou em modo reduzido (sem GPU) —'
          ' os números acima validam o fluxo, não são resultado final.')


OK — persistido em: /content/drive/MyDrive/deep-acordao-tcu2


In [6]:
from pathlib import Path
for f in sorted(Path('/content/drive/MyDrive/deep-acordao-tcu2/resultados').glob('*.json')):
    print(f'=== {f.name} ===')
    print(f.read_text())
    print()

=== metricas_baseline.json ===
{
  "representacao": "TF-IDF 50k bigramas + LogReg",
  "feature": "VOTO_LIMPO (sem SUMARIO, sem dispositivo, veredito residual mascarado)",
  "kfold_5x_balanced": {
    "mean_f1": 0.4909779582966717,
    "std_f1": 0.07045756673317659,
    "ci_95": [
      0.40349333428304696,
      0.5784625823102965
    ],
    "mean_acc": 0.912458734680957,
    "per_class_f1": {
      "Irregular": 0.9579103506875711,
      "Regular com Ressalva": 0.3990528282317481,
      "Regular": 0.11597069597069598
    },
    "confusion_matrix": [
      [
        3231,
        75,
        4
      ],
      [
        150,
        87,
        4
      ],
      [
        55,
        31,
        7
      ]
    ]
  },
  "kfold_5x_sem_pesos": {
    "mean_f1": 0.3173233893115558,
    "per_class_f1": {
      "Irregular": 0.9519701679346673,
      "Regular com Ressalva": 0.0,
      "Regular": 0.0
    }
  },
  "holdout_temporal_2024": {
    "f1_macro": 0.41218442574374775,
    "accuracy": 0.87022